In [ ]:
import numpy as np
import tensorflow as tf
import matplotlib.pyplot as plt
import seaborn as sns

from tensorflow.keras.applications import MobileNetV2
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.layers import GlobalAveragePooling2D, Reshape, LSTM, Dense, Dropout, Bidirectional
from tensorflow.keras.models import Model
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.callbacks import EarlyStopping

from sklearn.metrics import classification_report, confusion_matrix, roc_curve, auc, accuracy_score

In [ ]:
data_dir = "/kaggle/input/datasets/afridirahman/brain-stroke-ct-image-dataset/Brain_Data_Organised"

In [ ]:
img_size = (224, 224)
batch_size = 32

datagen = ImageDataGenerator(
    rescale=1./255,
    validation_split=0.2,
    rotation_range=20,
    zoom_range=0.2,
    horizontal_flip=True
)

train_data = datagen.flow_from_directory(
    data_dir,
    target_size=img_size,
    batch_size=batch_size,
    class_mode='binary',
    subset='training'
)

val_data = datagen.flow_from_directory(
    data_dir,
    target_size=img_size,
    batch_size=batch_size,
    class_mode='binary',
    subset='validation',
    shuffle=False
)

In [ ]:
class_weights = {0:1.0, 1:2.0}

In [ ]:
def build_bilstm_model(units=64, stacked=False, dropout=False):

    base_model = MobileNetV2(weights='imagenet', include_top=False, input_shape=(224,224,3))

    # Freeze some layers
    for layer in base_model.layers[:-15]:
        layer.trainable = False

    x = base_model.output
    x = GlobalAveragePooling2D()(x)

    # Convert to sequence
    x = Reshape((32, -1))(x)

    # 🔥 BiLSTM
    if stacked:
        x = Bidirectional(LSTM(units, return_sequences=True))(x)
        x = Bidirectional(LSTM(units))(x)
    else:
        x = Bidirectional(LSTM(units))(x)

    # Optional Dropout
    if dropout:
        x = Dropout(0.5)(x)

    # Dense layers
    x = Dense(256, activation='relu')(x)
    x = Dense(128, activation='relu')(x)

    output = Dense(1, activation='sigmoid')(x)

    model = Model(inputs=base_model.input, outputs=output)

    model.compile(
        optimizer=Adam(learning_rate=0.0001),
        loss='binary_crossentropy',
        metrics=['accuracy']
    )

    return model

In [ ]:
model_bilstm_single = build_bilstm_model(units=64, stacked=False)

history_single = model_bilstm_single.fit(
    train_data,
    validation_data=val_data,
    epochs=20,
    class_weight=class_weights,
    callbacks=[EarlyStopping(patience=5, restore_best_weights=True)]
)

In [ ]:
model_bilstm_stacked = build_bilstm_model(units=128, stacked=True)

history_stacked = model_bilstm_stacked.fit(
    train_data,
    validation_data=val_data,
    epochs=20,
    class_weight=class_weights,
    callbacks=[EarlyStopping(patience=5, restore_best_weights=True)]
)

In [ ]:
model_bilstm_stacked = build_bilstm_model(units=128, stacked=True)

history_stacked = model_bilstm_stacked.fit(
    train_data,
    validation_data=val_data,
    epochs=20,
    class_weight=class_weights,
    callbacks=[EarlyStopping(patience=5, restore_best_weights=True)]
)

In [ ]:
def evaluate_model(model, val_data):

    y_pred_prob = model.predict(val_data)
    y_pred = (y_pred_prob > 0.5).astype(int)
    y_true = val_data.classes

    print("Accuracy:", accuracy_score(y_true, y_pred))
    print(classification_report(y_true, y_pred))

    cm = confusion_matrix(y_true, y_pred)
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues')
    plt.show()

    fpr, tpr, _ = roc_curve(y_true, y_pred_prob)
    roc_auc = auc(fpr, tpr)

    print("AUC:", roc_auc)

    plt.plot(fpr, tpr, label='AUC = %.3f' % roc_auc)
    plt.plot([0,1], [0,1], 'r--')
    plt.legend()
    plt.show()

In [ ]:
evaluate_model(model_bilstm_single, val_data)
evaluate_model(model_bilstm_stacked, val_data)
evaluate_model(model_bilstm_dropout, val_data)